In [ ]:
from collections.abc import Iterable

import matplotlib.pyplot as plt
import numpy as np
import scipp as sc
from ipywidgets import widgets, Layout
from plopp.backends.matplotlib.utils import make_figure

from vector_alignment import Goniometer, GoniometerAxis

In [ ]:
%matplotlib widget

In [ ]:
class Quantity(widgets.HBox):
    def __init__(self, *, units: Iterable[str], default_unit:str|None=None,
                description:str="", value:float=0.0, layout:Layout|None=None) -> None:
        number, unit = _make_quantity_inputs(
            units=units,
            default_unit=default_unit,
            description=description,
            value=value,
        )
        self._number = number
        self._unit = unit
        super().__init__((number, unit), layout=layout or Layout(display='flex', flex_flow='row', min_width='0'))

    @property
    def value(self) -> sc.Variable:
        return sc.scalar(self._number.value, unit=self._unit.value)

def _make_quantity_inputs(*, units: Iterable[str], default_unit:str|None,
                description:str, value:float)->tuple[widgets.FloatText, widgets.Dropdown]:
    units = list(units)
    if default_unit is None:
        default_unit = units[0]

    number = widgets.FloatText(
        value=value,
        description=description,
        layout=widgets.Layout(flex='1 1 auto', width='auto', min_width='0'),
        style={"description_width": "initial"},
    )

    # Width just wide enough for the longest option (plus room for the
    # dropdown arrow / padding). ~0.62em per character is a decent estimate
    # for typical UI fonts; add a fixed padding for the arrow/border.
    longest = max((len(u) for u in units), default=1)
    dropdown_width = f"{longest * 0.62 + 1.2}em"

    unit = widgets.Dropdown(
        options=units,
        value=default_unit,
        layout=widgets.Layout(width=dropdown_width, flex='0 0 auto'),
    )
    unit.add_class('sc-no-chevron')

    state = {"previous_unit": unit.value}
    def _on_unit_change(change):
        if change["name"] != "value":
            return
        old_unit = state["previous_unit"]
        new_unit = change["new"]
        state["previous_unit"] = new_unit

        try:
            number.value = sc.to_unit(sc.scalar(number.value, unit=old_unit), new_unit).value
        except Exception as exc:
            # Don't let a bad conversion crash the widget; surface the error.
            print(f"Unit conversion failed ({old_unit} -> {new_unit}): {exc}")

    unit.observe(_on_unit_change, names="value")

    return number, unit

In [ ]:
label_layout = Layout(flex='0 0 auto', width='auto', padding='0 0 0 2em')
flex_item_layout = Layout(flex='1 1 auto', width='auto', min_width='0')
up_down_button_layout = Layout(
    flex='0 0 auto',
    width='auto',
    height='13px',
    padding='0',
    margin='0',
    align_self='stretch',
)
_STYLE = """<style>
.sc-tiny-btn {
    display: flex !important;
    align-items: center;
    justify-content: center;
    line-height: 1 !important;
    padding: 0 !important;

    border-radius: 0 !important;
}
.sc-tiny-btn .fa, .sc-tiny-btn i {
    margin: 0 !important;
    font-size: 8px !important;
}
.sc-no-chevron select {
    appearance: none !important;
    -webkit-appearance: none !important;
    -moz-appearance: none !important;
    background-image: none !important;
    padding-right: 0.4em !important;   /* reclaim space the arrow used */
}
</style>"""


# TODO make it work for static backend. (Plopp has something)
class GoniometerWidget(widgets.VBox):
    def __init__(self)->None:
        self._axes = GoniometerAxesWidget(self._on_update)
        self._fig = _prepare_goniometer_plot()
        super().__init__(
            (widgets.HTML(_STYLE), widgets.HBox((self._axes, self._fig.canvas))),
            layout=Layout(width='100%')
        )

    @property
    def value(self) -> Goniometer:
        return Goniometer(
            axes=self._axes.value
        )
    
    def _on_update(self):
        ax = self._fig.get_axes()[0]
        ax.cla()
        self.value.plot(ax=ax)
        self._fig.canvas.draw_idle()

def _prepare_goniometer_plot():
    fig = make_figure(figsize=(3, 3), layout="constrained")
    fig.canvas.header_visible = False
    # Resizing acts strangely when the figure is part of a flex layout.
    # Make it non-resizable to avoid users making it unreadable.
    # We may want to keep it resizable if the default size does not turn out well.
    fig.canvas.resizable = False
    
    ax = fig.add_subplot(projection='3d')
    ax.format_coord = lambda *args, **kwargs: ""
    Goniometer(axes=[]).plot(ax=ax)
    return fig

class GoniometerAxesWidget(widgets.VBox):
    def __init__(self, on_update):
        self._add_axis_button = widgets.Button(description="Add axis", icon="plus")
        self._add_axis_button.on_click(lambda *args: self.add_axis())
        self._axes = []
        self._on_update = on_update

        super().__init__((self._add_axis_button,), layout=Layout(width='100%'))

    @property
    def value(self) -> list[GoniometerAxis]:
        return [
            axis.value for axis in self._axes]

    def add_axis(self) -> None:
        axis = GoniometerAxisWidget(move_row=self._move_row, remove_row=self._remove_row)
        _observe_deep(axis, lambda change: self._on_update())
        self._axes.insert(0, axis)
        self._set_children()
        self._on_update()

    def _move_row(self, button, axis):
        idx = self._axes.index(axis)
        new_idx = idx + button._move
        if 0 <= new_idx < len(self._axes):
            self._axes[idx], self._axes[new_idx] = self._axes[new_idx], self._axes[idx]
            self._set_children()
            self._on_update()

    def _remove_row(self, axis):
        self._axes.remove(axis)
        self._set_children()
        self._on_update()
        

    def _set_children(self):
        self.children = (*self._axes, self._add_axis_button)
        
       

def _observe_deep(widget, handler, names='value'):
    if widget.has_trait(names):
        widget.observe(handler, names=names)
    for child in getattr(widget, 'children', ()):
        _observe_deep(child, handler, names)


class GoniometerAxisWidget(widgets.HBox):
    def __init__(self, *, move_row, remove_row):
        children, inputs = _make_goniometer_axis_row(lambda button: move_row(button, self),
                                                    lambda button: remove_row(self))
        super().__init__(children, layout=Layout(
            width='100%',
            display='flex', flex_flow='row', align_items='center'))
        
        self._inputs = inputs

    @property
    def value(self) -> GoniometerAxis:
        ax = self._inputs["axis"].value
        axis = sc.vector([
            1 * ("X" in ax),
            1 * ("Y" in ax),
            1 * ("Z" in ax),
        ]) * (-1 if "-" in ax else +1)
        return GoniometerAxis(
            axis=axis,
            name=self._inputs["name"].value,
            min=self._inputs["lo"].value,
            max=self._inputs["hi"].value,  # TODO rename to min, max
        )


def _make_goniometer_axis_row(move_row, remove_row):
    inputs = {}
    
    inputs['name'] = name = widgets.Text(placeholder="Name", layout=flex_item_layout)
    axis_label = widgets.Label(value="Axis", layout=label_layout)
    inputs['axis'] = axis = widgets.Dropdown(
        options=("+X", "-X", "+Y", "-Y", "+Z", "-Z"),
        value="+Y",
        description="Axis",
        layout=widgets.Layout(flex='1 1 auto', width='auto', min_width='11em'),
    )
    limits_label = widgets.Label(value="Limits", layout=label_layout)
    # TODO inf or empty
    # inputs['lo'] = lo = Quantity(value=-np.inf, units=('deg', 'rad'), layout=flex_item_layout)
    # inputs['hi'] = hi = Quantity(value=np.inf, units=('deg', 'rad'), layout=flex_item_layout)
    inputs['lo'] = lo = Quantity(value=0, units=('deg', 'rad'), layout=flex_item_layout)
    inputs['hi'] = hi = Quantity(value=0, units=('deg', 'rad'), layout=flex_item_layout)

    up_btn = widgets.Button(icon='caret-up', layout=up_down_button_layout)
    down_btn = widgets.Button(icon='caret-down', layout=up_down_button_layout)
    up_btn.add_class('sc-tiny-btn')
    down_btn.add_class('sc-tiny-btn')
    up_btn._move = -1
    down_btn._move = 1
    up_btn.on_click(move_row)
    down_btn.on_click(move_row)
    up_down_buttons = widgets.VBox((up_btn, down_btn),
                          layout=Layout(width='30px', height='28px', flex='0 0 auto', justify_content='space-between')
            )

    remove_button = widgets.Button(icon='trash', layout=Layout(
            flex='0 0 auto',
            width='auto',
            align_self='stretch',
        ))
    remove_button.on_click(remove_row)
    
    children =        (up_down_buttons, name, axis, limits_label, lo, hi, remove_button)
    return children, inputs


goni = GoniometerWidget()
goni

In [ ]:
goni.value.axes